In [2]:
!pip install -q --upgrade zarr numcodecs xarray dask


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
distributed 2025.9.1 requires dask==2025.9.1, but you have dask 2025.12.0 which is incompatible.
rapids-dask-dependency 25.10.0 requires dask==2025.9.1, but you have dask 2025.12.0 which is incompatible.


In [1]:
import xarray as xr
xr.backends.list_engines()


{'h5netcdf': <H5netcdfBackendEntrypoint>
   Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using h5netcdf in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.H5netcdfBackendEntrypoint.html,
 'scipy': <ScipyBackendEntrypoint>
   Open netCDF files (.nc, .cdf and .nc.gz) using scipy in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html,
 'store': <StoreBackendEntrypoint>
   Open AbstractDataStore instances in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html,
 'zarr': <ZarrBackendEntrypoint>
   Open zarr files (.zarr) using zarr in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ZarrBackendEntrypoint.html}

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import xarray as xr
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

DATA_PATH = "/content/drive/MyDrive/Indian Weather Prediction/datasets/processed.zarr"
MODEL_PATH = "/content/drive/MyDrive/Indian Weather Prediction/models/convlstm_final.pt"

VARS = ["t2m", "d2m", "sp", "msl", "u10", "v10", "tp"]
INPUT_STEPS = 4
BATCH_SIZE = 4
EPOCHS = 6

# -------------------------
# MODEL
# -------------------------
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(7, 32, 3, padding=1)
        self.relu = nn.ReLU()
        self.temp_head = nn.Conv2d(32, 1, 1)
        self.rain_head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        x = x[:, -1]
        x = self.relu(self.conv(x))
        return self.temp_head(x), self.rain_head(x)

model = ConvNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
scaler = GradScaler()

# -------------------------
# LOAD DATASET HANDLE
# -------------------------
ds = xr.open_zarr(DATA_PATH)

train_years = ["2019", "2020", "2021"]  # keep it light & fast

print("Starting training...")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for year in train_years:
        print(f"Epoch {epoch+1} | Loading year {year}")

        ds_y = ds.sel(valid_time=slice(f"{year}-01-01", f"{year}-12-31"))

        data = np.stack([ds_y[v].values for v in VARS], axis=1)
        targets_t = ds_y["t2m"].values
        targets_r = ds_y["tp"].values

        X, Yt, Yr = [], [], []
        for i in range(len(data) - INPUT_STEPS):
            X.append(data[i:i+INPUT_STEPS])
            Yt.append(targets_t[i+INPUT_STEPS])
            Yr.append(targets_r[i+INPUT_STEPS])

        X = torch.tensor(np.array(X), dtype=torch.float32)
        Yt = torch.tensor(np.array(Yt), dtype=torch.float32)
        Yr = torch.tensor(np.array(Yr), dtype=torch.float32)

        loader = DataLoader(
            TensorDataset(X, Yt, Yr),
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        for x, yt, yr in loader:
            x, yt, yr = x.to(device), yt.to(device), yr.to(device)
            optimizer.zero_grad()

            with autocast(device_type="cuda"):
                pt, pr = model(x)
                loss = loss_fn(pt.squeeze(), yt) + loss_fn(pr.squeeze(), yr)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()

    print(f"Epoch {epoch+1} completed | Loss: {epoch_loss:.4f}")

torch.save(model.state_dict(), MODEL_PATH)
print("✅ Model trained and saved:", MODEL_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Starting training...
Epoch 1 | Loading year 2019
Epoch 1 | Loading year 2020
Epoch 1 | Loading year 2021
Epoch 1 completed | Loss: 101.3601
Epoch 2 | Loading year 2019
Epoch 2 | Loading year 2020
Epoch 2 | Loading year 2021
Epoch 2 completed | Loss: 86.4478
Epoch 3 | Loading year 2019
Epoch 3 | Loading year 2020
Epoch 3 | Loading year 2021
Epoch 3 completed | Loss: 83.6795
Epoch 4 | Loading year 2019
Epoch 4 | Loading year 2020
Epoch 4 | Loading year 2021
Epoch 4 completed | Loss: 82.4667
Epoch 5 | Loading year 2019
Epoch 5 | Loading year 2020
Epoch 5 | Loading year 2021
Epoch 5 completed | Loss: 82.2805
Epoch 6 | Loading year 2019
Epoch 6 | Loading year 2020
Epoch 6 | Loading year 2021
Epoch 6 completed | Loss: 81.2243
✅ Model trained and saved: /content/drive/MyDrive/Indian Weather Prediction/models/convlstm_final.pt


In [ ]:
!nvidia-smi
